# HOG size vs. log2FC. 

Load packages

In [11]:
import numpy as np
import pandas as pd
import plotly.express as px

Load the annotated result dataset from Salmon map

In [12]:
salmon_map_full_annot = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_map_dominance_DE_sex_results_new_filtering.csv", float_precision='legacy')

# Replace the zeros in padj with 1e-308 avoid log10 issues
salmon_map_full_annot["padj_safe"] = salmon_map_full_annot["padj"].replace(0, 1e-308).fillna(1)

# Add the negative log 10 padj for plotting
salmon_map_full_annot["neglog10_padj"] = -np.log10(salmon_map_full_annot["padj_safe"])

# Add significance to differentially expressed genes 
salmon_map_full_annot["significant"] = (
    (salmon_map_full_annot["padj"] < 0.05) &
    (salmon_map_full_annot["log2FoldChange"].abs() > 1)
)

# Add a label to the significant genes
salmon_map_full_annot["label"] = salmon_map_full_annot["gene_id"].where(salmon_map_full_annot["significant"], "")

salmon_map_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,GOs,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs,padj_safe,neglog10_padj,significant,label
0,248.420188,0.692250,0.131096,5.280476,1.288487e-07,2.675948e-07,g2.t1,g2,utg000001l,220384.0,...,-,-,ko:K02273,"ko00190,ko01100,ko04260,ko04714,ko04932,ko0501...",I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",2.675948e-07,6.572522,False,
1,92.726112,0.655815,0.169749,3.863442,1.118004e-04,1.938610e-04,g3.t1,g3,utg000001l,227675.0,...,"GO:0003674,GO:0005488,GO:0005515,GO:0005543,GO...",3.1.26.5,"ko:K14529,ko:K17543","ko03013,map03013",DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",1.938610e-04,3.712510,False,
2,136.044931,-0.620454,0.178733,-3.471396,5.177602e-04,8.480073e-04,g4.t1,g4,utg000001l,245866.0,...,"GO:0001101,GO:0001666,GO:0003674,GO:0003824,GO...",2.5.1.61,ko:K01749,"ko00860,ko01100,ko01110,ko01120,map00860,map01...",H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",8.480073e-04,3.071600,False,
3,3.332468,4.516260,0.827360,5.458642,4.797891e-08,1.019443e-07,g5.t1,g5,utg000001l,257697.0,...,"GO:0005575,GO:0005622,GO:0005623,GO:0005737,GO...",-,-,-,I,"KOG2761@1|root,KOG2761@2759|Eukaryota,38F7R@33...",1.019443e-07,6.991637,True,g5
4,381.084979,-0.916813,0.069597,-13.173185,1.252016e-39,7.438447e-39,g6.t1,g6,utg000001l,263441.0,...,-,-,-,-,B,"2E6V3@1|root,2SDHR@2759|Eukaryota",7.438447e-39,38.128518,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17569,20.275098,0.380247,0.209991,1.810783,7.017444e-02,8.988015e-02,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,NaN,NaN,NaN,NaN,8.988015e-02,1.046336,False,
17570,1.364118,2.296660,0.615847,3.729268,1.920367e-04,3.261357e-04,g34884.t1,g34884,utg003700l,19710.0,...,-,-,-,-,S,"2D3G7@1|root,2SRFN@2759|Eukaryota,3AMW9@33154|...",3.261357e-04,3.486602,True,g34884
17571,8.286716,2.238529,0.365104,6.131210,8.721322e-10,2.014570e-09,g34922.t1,g34922,utg003714l,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.014570e-09,8.695818,True,g34922
17572,19.121063,2.377057,0.337972,7.033304,2.016994e-12,5.208913e-12,g35167.t1,g35167,utg003885l,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.208913e-12,11.283253,True,g35167


Remove entries that does not have a HOG

In [13]:
salmon_map_results_HOG = salmon_map_full_annot.dropna(subset=["HOG"])


Compute number of paralogs in each HOG

In [14]:
hog_size = (
    salmon_map_results_HOG
    .groupby("HOG")
    .size()
    .reset_index(name="HOG_size")
)

Add the hog_size column to the results table by merging on HOG name

In [15]:
salmon_map_results_HOG = salmon_map_results_HOG.merge(
    hog_size,
    on="HOG",
    how="left"
)

salmon_map_results_HOG
# Down from 14.164 to 12.837 transcripts

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs,padj_safe,neglog10_padj,significant,label,HOG_size
0,248.420188,0.692250,0.131096,5.280476,1.288487e-07,2.675948e-07,g2.t1,g2,utg000001l,220384.0,...,-,ko:K02273,"ko00190,ko01100,ko04260,ko04714,ko04932,ko0501...",I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",2.675948e-07,6.572522,False,,2
1,92.726112,0.655815,0.169749,3.863442,1.118004e-04,1.938610e-04,g3.t1,g3,utg000001l,227675.0,...,3.1.26.5,"ko:K14529,ko:K17543","ko03013,map03013",DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",1.938610e-04,3.712510,False,,2
2,136.044931,-0.620454,0.178733,-3.471396,5.177602e-04,8.480073e-04,g4.t1,g4,utg000001l,245866.0,...,2.5.1.61,ko:K01749,"ko00860,ko01100,ko01110,ko01120,map00860,map01...",H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",8.480073e-04,3.071600,False,,2
3,3.332468,4.516260,0.827360,5.458642,4.797891e-08,1.019443e-07,g5.t1,g5,utg000001l,257697.0,...,-,-,-,I,"KOG2761@1|root,KOG2761@2759|Eukaryota,38F7R@33...",1.019443e-07,6.991637,True,g5,2
4,381.084979,-0.916813,0.069597,-13.173185,1.252016e-39,7.438447e-39,g6.t1,g6,utg000001l,263441.0,...,-,-,-,B,"2E6V3@1|root,2SDHR@2759|Eukaryota",7.438447e-39,38.128518,False,,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15643,10.439363,0.155823,1.108231,0.140605,8.881818e-01,9.024054e-01,g34647.t1,g34647,utg003542l,22333.0,...,-,-,-,T,"COG4886@1|root,KOG4641@2759|Eukaryota,39T87@33...",9.024054e-01,0.044598,False,,8
15644,20.275098,0.380247,0.209991,1.810783,7.017444e-02,8.988015e-02,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,NaN,NaN,NaN,8.988015e-02,1.046336,False,,3
15645,1.364118,2.296660,0.615847,3.729268,1.920367e-04,3.261357e-04,g34884.t1,g34884,utg003700l,19710.0,...,-,-,-,S,"2D3G7@1|root,2SRFN@2759|Eukaryota,3AMW9@33154|...",3.261357e-04,3.486602,True,g34884,8
15646,8.286716,2.238529,0.365104,6.131210,8.721322e-10,2.014570e-09,g34922.t1,g34922,utg003714l,1.0,...,NaN,NaN,NaN,NaN,NaN,2.014570e-09,8.695818,True,g34922,8


Plot each transcript vs. HOG size 

In [16]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

jitter = np.random.uniform(
    -0.15, 0.15,
    size=len(salmon_map_results_HOG)
)

fig = px.scatter(
    x=salmon_map_results_HOG["HOG_size"] + jitter,
    y=salmon_map_results_HOG["log2FoldChange"],
    opacity=0.4,
    labels={
        "x": "HOG size (number of paralogs)",
        "y": "Transcript log2FoldChange"
    },
    title="Transcript-level sex bias as a function of HOG size"
)

fig.add_hline(y=0, line_dash="dash")
fig.show()


Try with absolute values instead of directional.  
Add abs

In [17]:
salmon_map_results_HOG["abs_log2FC"] = (
    salmon_map_results_HOG["log2FoldChange"].abs()
)

Plot abs bias vs. HOG size 

In [18]:
fig = px.scatter(
    salmon_map_results_HOG,
    x=salmon_map_results_HOG["HOG_size"] + jitter,
    y=salmon_map_results_HOG["log2FoldChange"],
    hover_name="transcript_id",
    opacity=0.4,
    labels={
        "x": "HOG size (number of paralogs)",
        "y": "Transcript log2FoldChange"
    },
    title="Transcript-level absolute sex bias vs. HOG size"
)

fig.show()


Variance within each HOG size group. Group by HOG_size instead of HOG

In [19]:
variance_by_size = (
    salmon_map_results_HOG
    .groupby("HOG_size")["log2FoldChange"]
    .agg(
        variance="var",
        sd="std",
        n="count"
    )
    .reset_index()
)
variance_by_size


,HOG_size,variance,sd,n
0,1,4.128091,2.031770,8173
1,2,5.951122,2.439492,3726
2,3,6.416234,2.533029,1194
3,4,7.557789,2.749143,672
4,5,5.582964,2.362830,415
5,6,3.596792,1.896521,306
6,7,5.407085,2.325314,182
7,8,7.520620,2.742375,240
8,9,5.432133,2.330694,162
9,10,3.351369,1.830674,60


Plot variance vs HOG Size 

In [21]:
fig = px.line(
    variance_by_size,
    x="HOG_size",
    y="sd",
    markers=True,
    labels={
        "HOG_size": "HOG size (number of paralogs)",
        "sd": "SD of log2FoldChange"
    },
    title="Variance in sex-biased expression amaon paralogs by HOG size"
)

fig.show()
#Do within hogs, plot against hog size. 

# Blocks below for averages within HOGs:

Compute mean sex bias in each HOG

In [22]:
#Do absolute instead of directional? 
hog_lfc = (
    salmon_map_results_HOG
    .groupby("HOG")["log2FoldChange"]
    .mean()
    .reset_index(name="mean_log2FC")
)

Merge

In [23]:
hog_summary = hog_lfc.merge(hog_size, on="HOG")
hog_summary

,HOG,mean_log2FC,HOG_size
0,N0.HOG0000007,1.886425,17
1,N0.HOG0000008,1.805293,17
2,N0.HOG0000009,1.760048,14
3,N0.HOG0000010,0.833681,25
4,N0.HOG0000011,-1.495594,4
...,...,...,...
10845,N0.HOG0024543,0.481831,2
10846,N0.HOG0024547,-0.152003,1
10847,N0.HOG0024667,0.447413,1
10848,N0.HOG0024673,2.179755,1


Plot

In [24]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_summary["HOG_size_jitter"] = (
    hog_summary["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_summary))
)


fig = px.scatter(
    hog_summary,
    x="HOG_size_jitter",
    y="mean_log2FC",
    hover_name="HOG",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size (number of paralogs)",
        "mean_log2FC": "Mean log2FoldChange"
    },
    title="Paralog number vs sex-biased expression (Directional)"
)

fig.add_hline(y=0, line_dash="dash")
fig.show()


Try with absolute L2FC

In [ ]:
hog_lfc_abs = (
    salmon_map_results_HOG
    .assign(abs_log2FC=lambda x: x["log2FoldChange"].abs())
    .groupby("HOG")["abs_log2FC"]
    .mean()
    .reset_index(name="mean_abs_log2FC")
)

hog_abs_summary = hog_lfc_abs.merge(hog_size, on="HOG")


In [ ]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_abs_summary["HOG_size_jitter"] = (
    hog_abs_summary["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_abs_summary))
)

fig = px.scatter(
    hog_abs_summary,
    x="HOG_size_jitter",
    y="mean_abs_log2FC",
    hover_name="HOG",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size (number of paralogs)",
        "mean_abs_log2FC": "Mean |log2FoldChange|"
    },
    title="Paralog number vs strength of sex-biased expression (Absolute)"
)

fig.show()


# Male-bias only and female-bias only

In [ ]:
#split each sex up based on log2fc
male_biased = salmon_map_results_HOG[
    salmon_map_results_HOG["log2FoldChange"] > 0
]

female_biased = salmon_map_results_HOG[
    salmon_map_results_HOG["log2FoldChange"] < 0
]


In [ ]:
# male-biased
hog_male = (
    male_biased
    .groupby("HOG")["log2FoldChange"]
    .mean()
    .reset_index(name="mean_male_log2FC")
)


In [ ]:
#female biased
hog_female = (
    female_biased
    .assign(abs_log2FC=lambda x: x["log2FoldChange"].abs())
    .groupby("HOG")["abs_log2FC"]
    .mean()
    .reset_index(name="mean_female_log2FC")
)


In [ ]:
#hog size same as before
hog_size = (
    salmon_map_results_HOG
    .groupby("HOG")
    .size()
    .reset_index(name="HOG_size")
)


In [ ]:
#merge all
hog_sex_bias = (
    hog_size
    .merge(hog_male, on="HOG", how="left")
    .merge(hog_female, on="HOG", how="left")
)


hog_sex_bias


,HOG,HOG_size,mean_male_log2FC,mean_female_log2FC
0,N0.HOG0000007,8,1.660752,NaN
1,N0.HOG0000008,8,1.131950,0.408400
2,N0.HOG0000009,4,1.453381,NaN
3,N0.HOG0000010,12,0.689358,0.495909
4,N0.HOG0000011,1,NaN,1.429319
...,...,...,...,...
9608,N0.HOG0024541,2,NaN,1.486879
9609,N0.HOG0024543,2,0.481861,NaN
9610,N0.HOG0024547,1,NaN,0.151882
9611,N0.HOG0024673,1,2.180027,NaN


plot male biased transcripts

In [ ]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_sex_bias["HOG_size_jitter"] = (
    hog_sex_bias["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_sex_bias))
)

fig = px.scatter(
    hog_sex_bias,
    x="HOG_size_jitter",
    y="mean_male_log2FC",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size",
        "mean_male_log2FC": "Mean male-biased log2FC"
    },
    title="Paralog number vs male-biased expression strength"
)
fig.show()


Plot female baised transcripts

In [ ]:
fig = px.scatter(
    hog_sex_bias,
    x="HOG_size_jitter",
    y="mean_female_log2FC",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size",
        "mean_female_log2FC": "Mean female-biased |log2FC|"
    },
    title="Paralog number vs female-biased expression strength"
)
fig.show()
